In [15]:
import requests

URL = "https://apim-openapi-gbgco-prod.azure-api.net/gql"
KEY = "fe43d10b3be341f59f59ec98fb3bd29c"


def gql(query):
    r = requests.post(
        URL,
        json={"query": query},
        headers={
            "Ocp-Apim-Subscription-Key": KEY,
            "Content-Type": "application/json",
        },
        timeout=30,
    )
    if not r.ok:
        raise RuntimeError(f"{r.status_code}: {r.text}")
    data = r.json()
    if "errors" in data:
        raise RuntimeError(data["errors"])
    return data["data"]


# Need revision
result = gql("""
query {
  guides(filter: { lang: "en", per_page: 10, page: 10 }) {
    guides {
      id
      title
      excerpt
      date
    }
  }
}
""")

for g in result["guides"]["guides"]:
    print(g["id"], g["title"])

46888 Summer in Gothenburg
51265 Sustainable experiences in and outside Gothenburg
57571 Spas in Gothenburg
57952 Gothenburg Art Map
58101 Christmas in Gothenburg
59015 Six trips just outside Gothenburg – castles, fika and the archipelago
65809 Christmas buffets in Gothenburg
66428 Conscious Christmas Shopping
69376 Atmospheric Christmas markets in the Gothenburg region
72529 New Year&#8217;s Eve in Gothenburg


In [ ]:
# EVENTS ETL -> events_raw.json + events.csv (read by the dashboard)
import html
import json
from pathlib import Path

import pandas as pd

OUT_DIR = Path.cwd()

# Without a date filter the API drops dates that have passed.
# With one it also returns earlier dates of events that are still listed.
DATE_FROM = "2020-01-01"
DATE_TO = "2035-12-31"

IMAGE = "width height source_url"
MEDIA = f"""
id credit caption alt_text media_type mime_type
sizes {{
  medium {{ {IMAGE} }}
  thumbnail {{ {IMAGE} }}
  large {{ {IMAGE} }}
  full {{ {IMAGE} }}
}}
"""

# Every field listed in the API docs for WpEntity
EVENT_FIELDS = f"""
id date modified type link title content excerpt lang
categories areas tags invisible_tags
translations {{ sv en de }}
featuredmedia {{ {MEDIA} }}
category_heading
gallery {{ {MEDIA} }}
contact {{ email phone website facebook instagram }}
location {{
  address lat lng zoom place_id name street_number
  street_name state post_code country country_short
}}
free
dates {{ start end }}
place_id
classification
currentInTime {{ months weekdays }}
"""


def fetch_events(lang, per_page=100):
    events, page = [], 1
    while True:
        data = gql(f"""
        query {{
          events(filter: {{
            lang: "{lang}", start: "{DATE_FROM}", end: "{DATE_TO}",
            per_page: {per_page}, page: {page}
          }}) {{
            events {{ {EVENT_FIELDS} }}
          }}
        }}
        """)
        batch = data["events"]["events"] or []
        events += batch
        print(f"{lang} page {page}: {len(batch)} events")
        if len(batch) < per_page:
            return events
        page += 1


# The en and sv feeds overlap: the same event exists in both languages.
# Keep the English version when there is one, plus Swedish-only events.
en = {e["id"]: e for e in fetch_events("en")}
sv = {e["id"]: e for e in fetch_events("sv")}

has_en = {(e.get("translations") or {}).get("sv") for e in en.values()}
raw = list(en.values()) + [
    e for i, e in sv.items()
    if i not in has_en and (e.get("translations") or {}).get("en") not in en
]

(OUT_DIR / "events_raw.json").write_text(
    json.dumps(raw, ensure_ascii=False, indent=1),
    encoding="utf-8",
)
print(f"{len(raw)} unique events ({len(en)} English, {len(raw) - len(en)} Swedish only)")

In [ ]:
# FLATTEN: one row per event date

# The API only gives category IDs without names, so a broad category is
# derived from the free-text category_heading (English or Swedish).
# First match wins.
CATEGORY_KEYWORDS = [
    ("Festival", ["festival"]),
    ("Market", ["market", "fair", "shopping",
                "marknad", "loppis", "mässa", "secondhand"]),
    ("Exhibition", ["exhibition", "museum", "photo", "gallery", "art ",
                    "illusion", "film screening", "artist",
                    "utställning", "konst", "foto", "visning", "film"]),
    ("Sports", ["match", "hockey", "football", "handball", "floorball",
                "marathon", "tournament", "sport", " race", "equestrian",
                "horse", "cup", "league", "athletics", "tennis", "bike",
                "championship", "competition", " vs",
                "fotboll", "handboll", "innebandy", "allsvenskan", "lopp",
                "turnering", "mästerskap", "tävling", "ridsport", "shl"]),
    ("Tours", ["boat", "archipelago", "cruise", "sightseeing", "guided",
               "båt", "skärgård", "kryssning", "guidad"]),
    ("Concert", ["concert", "music", "jazz", "opera", "choir", "orchestra",
                 "sing-along", "singer", "band", "rock", "tribute", "gig",
                 "konsert", "musik", "kör", "orkester", "sång", "visor",
                 "blues", "indie", "punk", "groove", "americana"]),
    ("Food & drink", ["buffet", "food", "wine", "beer", "tasting", "dinner",
                      "brunch", "lunch", "culinary", "flavour", "coffee",
                      "cheese", "all-you-can-eat",
                      "julbord", "matupplevelse", "vinprovning", "öl", "provning", "middag",
                      "fika", "kaffe"]),
    ("Theatre & shows", ["theatre", "theater", "stand-up", "standup",
                         "comedy", "dance", "ballet", "musical", "show",
                         "performance", "circus", "magic",
                         "teater", "föreställning", "komedi", "humor",
                         "dans", "balett", "cirkus", "revy", "musikal",
                         "dramatik"]),
    ("Tours", ["tour", "walk", "building",
               "vandring", "rundtur", "stadsvandring"]),
    ("Family", ["family", "children", "kids", "parent", "holiday",
                "break activities", "halloween", "amusement", "spooky",
                "lucia", "christmas",
                "familj", "barn", "höstlov", "sportlov", "sommarlov",
                "påsklov", "jul ", "julen", "julfirande", "påsk", "sagor"]),
]


def categorize(heading):
    text = (heading or "").lower()
    for category, words in CATEGORY_KEYWORDS:
        if any(w in text for w in words):
            return category
    return "Other"


def clean(text):
    return html.unescape(text or "").strip()


rows = []
for e in raw:
    loc = e.get("location") or {}
    contact = e.get("contact") or {}
    media = e.get("featuredmedia") or []
    image = (
        ((media[0].get("sizes") or {}).get("medium") or {}).get("source_url")
        if media else None
    )
    for d in e.get("dates") or []:
        if not d.get("start"):
            continue
        rows.append({
            "event_id": e["id"],
            "name": clean(e["title"]),
            "start": d["start"],
            "end": d.get("end"),
            "category": categorize(e.get("category_heading")),
            "event_type": clean(e.get("category_heading")) or "—",
            "venue": clean(loc.get("name")) or clean(loc.get("address")) or "Unknown",
            "address": clean(loc.get("address")),
            "lat": loc.get("lat"),
            "lon": loc.get("lng"),
            "free": bool(e.get("free")),
            "link": e.get("link"),
            "lang": e.get("lang"),
            "excerpt": clean(e.get("excerpt")),
            "image": image,
            "website": contact.get("website"),
            "published": e.get("date"),
            "modified": e.get("modified"),
            "category_ids": e.get("categories"),
            "area_ids": e.get("areas"),
            "tag_ids": e.get("tags"),
            "place_id": e.get("place_id"),
            "classification": e.get("classification"),
            "dates_listed": len(e.get("dates") or []),
        })

events = pd.DataFrame(rows)
events["start"] = pd.to_datetime(events["start"], errors="coerce")
events["end"] = pd.to_datetime(events["end"], errors="coerce")
events = events.dropna(subset=["start", "lat", "lon"])

events["year"] = events["start"].dt.year
events["month"] = events["start"].dt.month
events["day"] = events["start"].dt.day
events["weekday"] = events["start"].dt.day_name()

events = events.sort_values("start").reset_index(drop=True)
events.to_csv(OUT_DIR / "events.csv", index=False, encoding="utf-8")

print(f"{len(events)} event dates from {events['event_id'].nunique()} events")
print(events["category"].value_counts())
print(events.groupby(["year", "month"]).size())

In [ ]:
# RECURRING EVENTS: add estimated past editions
#
# The API keeps one listing per yearly event and overwrites it with the next
# edition (e.g. "Way Out West 2027", listed since 2020). Earlier editions are
# not in the API, so they are ESTIMATED here: same calendar dates, shifted
# back one year at a time, from the year the listing was created.
import re

YEAR_IN_TITLE = re.compile(r"\b20\d\d\b")

listed_since = {e["id"]: int(e["date"][:4]) for e in raw}

editions = events.groupby("event_id").agg(
    first=("start", "min"),
    last=("start", "max"),
    name=("name", "first"),
)
editions["edition_year"] = editions["first"].dt.year
editions["listed_since"] = editions.index.map(listed_since)
editions["days"] = (editions["last"] - editions["first"]).dt.days

# Concert and show listings with a year are usually one-off tours, so
# they only count when the name says festival, fair, gala, award etc.
ANNUAL_WORDS = r"fest|fair|mässa|gala|award|pris|cup|market|marknad|night|natta"
editions["category"] = events.groupby("event_id")["category"].first()
one_off_like = (
    editions["category"].isin(["Concert", "Theatre & shows"])
    & ~editions["name"].str.contains(ANNUAL_WORDS, case=False)
) | editions["name"].str.contains(
    r"tour|championship|mästerskap|euro ", case=False
)

# Yearly event = short (max 14 days), not a one-off tour, and either has a
# year in its title or has been listed for at least 2 years before this edition
recurring = editions[
    (editions["days"] <= 14)
    & ~one_off_like
    & (
        editions["name"].str.contains(YEAR_IN_TITLE)
        | (editions["edition_year"] - editions["listed_since"] >= 2)
    )
]

events["source"] = "API"
events["recurring"] = events["event_id"].isin(recurring.index)

copies = []
for event_id, r in recurring.iterrows():
    api_rows = events[events["event_id"] == event_id]
    for year in range(r["listed_since"], r["edition_year"]):
        back = r["edition_year"] - year
        c = api_rows.copy()
        c["start"] = c["start"] - pd.DateOffset(years=back)
        c["end"] = c["end"] - pd.DateOffset(years=back)
        c["name"] = c["name"].str.replace(
            YEAR_IN_TITLE, lambda m: str(int(m.group()) - back), regex=True
        )
        c["source"] = "Estimated past edition"
        copies.append(c)

events = pd.concat([events, *copies], ignore_index=True)
events["year"] = events["start"].dt.year
events["month"] = events["start"].dt.month
events["day"] = events["start"].dt.day
events["weekday"] = events["start"].dt.day_name()

events = events.sort_values("start").reset_index(drop=True)
events.to_csv(OUT_DIR / "events.csv", index=False, encoding="utf-8")

print(f"{len(recurring)} yearly events, {sum(len(c) for c in copies)} estimated dates added")
print(recurring.sort_values("listed_since")[["name", "listed_since", "edition_year"]].to_string())
print(events.groupby(["year", "source"]).size().unstack(fill_value=0))